# ⚡ 04. Generation 1 Baseline: SARIMAX Model

Classical statistical baseline with $(p,d,q)=(2,1,2)$ and seasonal order $(1,1,1,7)$.

In [ ]:
import pandas as pd
import numpy as np
from statsmodels.tsa.statespace.sarimax import SARIMAX

def train_and_predict_sarimax(train_df, test_df):
    train_daily = train_df['PJME_MW'].resample('D').mean()
    model = SARIMAX(
        train_daily,
        order=(2, 1, 2),
        seasonal_order=(1, 1, 1, 7),
        enforce_stationarity=False,
        enforce_invertibility=False,
    )
    fitted = model.fit(disp=False, maxiter=200)
    steps = max(1, len(test_df) // 24)
    forecast = fitted.get_forecast(steps=steps)
    pred_mean = forecast.predicted_mean
    ci = forecast.conf_int(alpha=0.20)
    
    # Upsample to hourly frequency
    q50 = pred_mean.resample('h').interpolate()[:len(test_df)]
    q10 = ci.iloc[:, 0].resample('h').interpolate()[:len(test_df)]
    q90 = ci.iloc[:, 1].resample('h').interpolate()[:len(test_df)]
    return q10.values, q50.values, q90.values
